In [ ]:
"""
DES grizY and Euclid VIS, Y, J, H bandpasses (absolute total system throughput).

Curves downloaded on first use into filter_curves/:
  DES    : speclite decam2014-{g,r,i,z,Y} (https://github.com/desihub/speclite), DECam total
           throughput, instrument + atmosphere at airmass 1.3. (Not the SVO CTIO/DECam DR1 curves:
           same shapes but ~1.7x higher, above the filter-only transmission in i and z, so not absolute.)
  Euclid : SVO Filter Profile Service (https://svo2.cab.inta-csic.es/svo/theory/fps/), Euclid/VIS.vis
           and Euclid/NISP.{Y,J,H}, full response (telescope + instrument + filter + detector)
"""
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table

plot_folder = "outputs/"
curve_folder = "filter_curves/"
URLS = {
    "svo": "https://svo2.cab.inta-csic.es/svo/theory/fps/getdata.php?format=ascii&id={}",
    "speclite": "https://raw.githubusercontent.com/desihub/speclite/main/speclite/data/filters/{}.ecsv",
}

# (survey, band label, source, filter ID)
FILTERS = [
    ("DES", "g", "speclite", "decam2014-g"),
    ("DES", "r", "speclite", "decam2014-r"),
    ("DES", "i", "speclite", "decam2014-i"),
    ("DES", "z", "speclite", "decam2014-z"),
    ("DES", "Y", "speclite", "decam2014-Y"),
    ("Euclid", "VIS", "svo", "Euclid/VIS.vis"),
    ("Euclid", r"$\mathbf{Y_E}$", "svo", "Euclid/NISP.Y"),
    ("Euclid", r"$\mathbf{J_E}$", "svo", "Euclid/NISP.J"),
    ("Euclid", r"$\mathbf{H_E}$", "svo", "Euclid/NISP.H"),
]
# One hue per survey (as in the footprint map), lighter to darker with wavelength
SURVEY_CMAPS = {"DES": "Oranges", "Euclid": "Blues"}
SURVEY_LABELS = {"DES": "DES", "Euclid": "Euclid (VIS + NISP)"}
# (DES (DECam, airmass 1.3)

def load_filter(source, filter_id):
    """Wavelength [nm] and throughput of a filter curve, downloaded on first use."""
    extension = ".ecsv" if source == "speclite" else ".dat"
    path = os.path.join(curve_folder, filter_id.replace("/", "_") + extension)
    if not os.path.exists(path):
        os.makedirs(curve_folder, exist_ok=True)
        urllib.request.urlretrieve(URLS[source].format(filter_id), path)
    if source == "speclite":
        table = Table.read(path, format="ascii.ecsv")  # wavelength in nm
        return np.asarray(table["wavelength"], float), np.asarray(table["response"], float)
    wavelength, throughput = np.loadtxt(path, unpack=True)  # wavelength in Angstrom
    return wavelength / 10, throughput


def label_band(ax, wave, throughput, band):
    """Band name above the curve, at its throughput-weighted centre."""
    weights = throughput * np.gradient(wave)  # the wavelength grid is not uniform
    centre = np.sum(wave * weights) / np.sum(weights)
    ax.text(centre, throughput.max() + 0.02, band, color="0.15",
            ha="center", va="bottom", fontsize=16, fontweight="bold")

fontsize_label = 12
fig, axes = plt.subplots(2, 1, figsize=(11, 7), dpi=150, sharex=True, sharey=True)
for ax, survey in zip(axes, SURVEY_CMAPS):
    bands = [(band, source, filter_id) for s, band, source, filter_id in FILTERS if s == survey]
    colors = plt.colormaps[SURVEY_CMAPS[survey]](np.linspace(0.5, 0.95, len(bands)))
    for (band, source, filter_id), color in zip(bands, colors):
        wave, throughput = load_filter(source, filter_id)
        ax.fill_between(wave, throughput, color=color, alpha=0.6, lw=0)
        ax.plot(wave, throughput, color=color, lw=1.2)
        label_band(ax, wave, throughput, band)
    ax.text(0.99, 0.95, SURVEY_LABELS[survey], transform=ax.transAxes,
            ha="right", va="top", fontsize=12, color="0.15")
    ax.set_ylabel("Throughput", fontsize=fontsize_label)
    ax.grid(alpha=0.25)

axes[0].set_xlim(350, 2100)
axes[0].set_ylim(0, 1.0)

axes[-1].set_xlabel("Wavelength [nm]", fontsize=fontsize_label)
fig.tight_layout()
os.makedirs(plot_folder, exist_ok=True)
fig.savefig(plot_folder + "DES_Euclid_bandpasses.png", bbox_inches="tight", dpi=500)
fig.savefig(plot_folder + "DES_Euclid_bandpasses.pdf", bbox_inches="tight", dpi=500)
plt.show()

In [ ]:
# Euclid and DES on the same plot: Euclid filled, DES as lines from blue to red with wavelength.
# (Uses FILTERS, load_filter and label_band from the cell above.) The DES colors are checked for
# color-blind readers: a dark green next to a light amber, so that r and i stay apart.
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

DES_COLORS = {"g": "#1f78b4", "r": "#1b7837", "i": "#e6a100", "z": "#d7301f", "Y": "#8c2d04"}

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)
euclid = [(band, source, filter_id) for s, band, source, filter_id in FILTERS if s == "Euclid"]
euclid_colors = plt.colormaps["Blues"](np.linspace(0.5, 0.95, len(euclid)))
for (band, source, filter_id), color in zip(euclid, euclid_colors):
    wave, throughput = load_filter(source, filter_id)
    ax.fill_between(wave, throughput, color=color, alpha=0.5, lw=0)
    ax.plot(wave, throughput, color=color, lw=1)
    label_band(ax, wave, throughput, band)

for survey, band, source, filter_id in FILTERS:
    if survey == "DES":
        wave, throughput = load_filter(source, filter_id)
        ax.plot(wave, throughput, color=DES_COLORS[band], lw=2)
        label_band(ax, wave, throughput, band)

ax.set_xlim(350, 2100)
ax.set_ylim(0, 1.0)
fontsize_label = 12
ax.set_xlabel("Wavelength [nm]", fontsize=fontsize_label)
ax.set_ylabel("Throughput", fontsize=fontsize_label)
ax.legend(
    handles=[
        Patch(facecolor=euclid_colors[1], alpha=0.5, label=SURVEY_LABELS["Euclid"]),
        Line2D([], [], color="0.3", lw=2, label=SURVEY_LABELS["DES"]),
    ],
    loc="upper right", frameon=False,
)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(plot_folder + "DES_Euclid_bandpasses_single.png", bbox_inches="tight", dpi=500)
fig.savefig(plot_folder + "DES_Euclid_bandpasses_single.pdf", bbox_inches="tight", dpi=500)
plt.show()